# Ingest sprints.json file
1. Read the all the files from the sprints folder using spark dataframe reader API
1. Define and enforce schema 
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../common/env_config

In [0]:
%run ../common/bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

#### Step 1 - Read the JSON file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

sprints_schema = StructType([
    StructField("date", StringType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])

In [0]:
# Read data from the sprints file
sprints_df = (
    spark.read
       .format('json')
       .schema(sprints_schema)
       .option('mode', 'FAILFAST')
       .option('multiLine', True)
       .load(source_file)
)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

#### Step 3 - Write to bronze delta table

In [0]:
write_to_bronze (
    input_df = sprints_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))